In [ ]:
import sys
import os

sys.path.append("..")
sys.path.append("../..")
sys.path.append("../../..")

In [ ]:
from datasets import load_dataset, load_from_disk
if not os.path.exists("../../data/criteo_attribution_dataset"):
    ds = load_dataset("criteo/criteo-attribution-dataset")

    ds.save_to_disk("../../data/criteo_attribution_dataset")

In [2]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, average_precision_score
import gc

In [ ]:
raw_df = load_from_disk("../../data/criteo_attribution_dataset")["train"].with_format("pandas")[:]
raw_df = raw_df.sort_values("timestamp").reset_index(drop=True)
raw_df.head(), raw_df.shape

(   timestamp       uid  campaign  conversion  conversion_timestamp  \
 0          0  20073966  22589171           0                    -1   
 1          2  24607497    884761           0                    -1   
 2          2  28474333  18975823           0                    -1   
 3          3   7306395  29427842           1               1449193   
 4          3  25357769  13365547           0                    -1   
 
    conversion_id  attribution  click  click_pos  click_nb  ...  \
 0             -1            0      0         -1        -1  ...   
 1             -1            0      0         -1        -1  ...   
 2             -1            0      0         -1        -1  ...   
 3        3063962            0      1          0         7  ...   
 4             -1            0      0         -1        -1  ...   
 
    time_since_last_click      cat1      cat2      cat3      cat4      cat5  \
 0                     -1   5824233   9312274   3490278  29196072  11409686   
 1        

In [4]:
raw_df = raw_df.dropna().reset_index(drop=True)
raw_df.head(), raw_df.shape

(   timestamp       uid  campaign  conversion  conversion_timestamp  \
 0          0  20073966  22589171           0                    -1   
 1          2  24607497    884761           0                    -1   
 2          2  28474333  18975823           0                    -1   
 3          3   7306395  29427842           1               1449193   
 4          3  25357769  13365547           0                    -1   
 
    conversion_id  attribution  click  click_pos  click_nb  ...  \
 0             -1            0      0         -1        -1  ...   
 1             -1            0      0         -1        -1  ...   
 2             -1            0      0         -1        -1  ...   
 3        3063962            0      1          0         7  ...   
 4             -1            0      0         -1        -1  ...   
 
    time_since_last_click      cat1      cat2      cat3      cat4      cat5  \
 0                     -1   5824233   9312274   3490278  29196072  11409686   
 1        

In [5]:
raw_df['is_first_click'] = (raw_df['time_since_last_click'] == -1).astype(int)

In [6]:
train_len = 0.7 * len(raw_df)
val_len = 0.1 * len(raw_df)

idx_train_end = int(train_len)
idx_val_end = int(train_len + val_len)

df_train = raw_df.iloc[:idx_train_end].copy()
df_val = raw_df.iloc[idx_train_end:idx_val_end].copy()
df_bidding = raw_df.iloc[idx_val_end:].copy()

In [7]:
def train_model_pair(df_train, df_val, config):
	print(f"Desc: {config['desc']}\n{'='*30}")

	selected_features = config['features']
	current_cat_features = config['cat_features']
	params = config['params']

	if config['sample_rate'] < 1.0:
		n_rows = int(len(df_train) * config['sample_rate'])
		df_t = df_train.iloc[-n_rows:]
	else:
		df_t = df_train

	print(f"Features: {len(selected_features)} | Cat features: {len(current_cat_features)}")
	print(f"Train rows: {len(df_t)}")

	metrics = {}

	print("--> Fitting CVR Model (Clicks only)...")

	df_t_clicks = df_t[df_t['click'] == 1]
	df_v_clicks = df_val[df_val['click'] == 1]

	train_pool = Pool(df_t_clicks[selected_features], df_t_clicks['conversion'], cat_features=current_cat_features)
	val_pool = Pool(df_v_clicks[selected_features], df_v_clicks['conversion'], cat_features=current_cat_features)

	model_cvr = CatBoostClassifier(**params)
	model_cvr.fit(train_pool, eval_set=val_pool)

	preds = model_cvr.predict_proba(df_v_clicks[selected_features])[:, 1]
	metrics['cvr_best_score'] = model_cvr.get_best_score()['validation']['Logloss']
	metrics['cvr_auc'] = roc_auc_score(df_v_clicks['conversion'], preds)
	metrics['cvr_pr'] = average_precision_score(df_v_clicks['conversion'], preds)

	del df_t_clicks, df_v_clicks, train_pool, val_pool, preds
	gc.collect()

	print("--> Fitting CTR Model...")

	train_pool = Pool(df_t[selected_features], df_t['click'], cat_features=current_cat_features)
	val_pool = Pool(df_val[selected_features], df_val['click'], cat_features=current_cat_features)

	model_ctr = CatBoostClassifier(**params)
	model_ctr.fit(train_pool, eval_set=val_pool)

	val_sub = df_val.sample(n=min(1_000_000, len(df_val)), random_state=42)
	preds = model_ctr.predict_proba(val_sub[selected_features])[:, 1]
	metrics['ctr_best_score'] = model_ctr.get_best_score()['validation']['Logloss']
	metrics['ctr_roc_auc'] = roc_auc_score(val_sub['click'], preds)
	metrics['ctr_pr_auc'] = average_precision_score(val_sub['click'], preds)

	del train_pool, val_pool, preds, val_sub, df_t
	gc.collect()
	
	return {
		'ctr_model': model_ctr,
		'cvr_model': model_cvr,
		'metrics': metrics
	}

In [8]:
ENSEMBLES_COUNT = 20

In [9]:
def apply_models_and_save(models_dict, df_bidding, name, config, filename="bidding_predictions.csv", ens_count=ENSEMBLES_COUNT):
    features = config['features']
    cat_features = config['cat_features']

    bidding_pool = Pool(df_bidding[features], cat_features=cat_features)

    for m_type in ['ctr', 'cvr']:
        print(f"--> Predicting {m_type} Uncertainty...")
        model = models_dict[f'{m_type}_model']

        virt_ens_preds = model.virtual_ensembles_predict(
            bidding_pool, 
            prediction_type='VirtEnsembles', 
            virtual_ensembles_count=ens_count,
        )

        mean_logit = virt_ens_preds.mean(axis=1)
        sigma_virt_ens_pred = virt_ens_preds.std(axis=1)

        mean_prob = 1 / (1 + np.exp(-mean_logit))
        sigma_data = mean_prob * (1 - mean_prob)

        df_bidding[f'{name}_{m_type}_pred_logit'] = mean_logit.astype(np.float32)
        df_bidding[f'{name}_{m_type}_logit_sigma'] = sigma_virt_ens_pred.astype(np.float32)
        df_bidding[f'{name}_{m_type}_sigma_data'] = sigma_data.astype(np.float32)

        del mean_logit, sigma_virt_ens_pred, mean_prob, sigma_data
        gc.collect()

    df_bidding.to_csv(filename, index=False)
    print(f"saved to {filename}")

    del bidding_pool
    gc.collect()

    return df_bidding

In [10]:
configs = {
	"all_features": {
		"desc": "All features, full data",
		"features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "time_since_last_click", "cat2", "cat4", "cat5", "is_first_click"],
		"cat_features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "cat2", "cat4", "cat5", "is_first_click"],
		"sample_rate": 1.0,
        "params": {
			'task_type': "CPU",
			'verbose': 200,
			'random_seed': 42,
			'allow_writing_files': False,

			'loss_function': 'Logloss',
			'eval_metric': 'Logloss',
			'boost_from_average': True,

			'iterations': 4000,
			'early_stopping_rounds': 300,
			'learning_rate': 2e-2,

			'depth': 6,
			'max_ctr_complexity': 4,
			'l2_leaf_reg': 5e2,
			'random_strength': 5.0,

			'posterior_sampling': True,
        }
	},
    "wo_1li": {
		"desc": "All \wo one less important, full data",
		"features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "time_since_last_click", "cat2", "cat4", "cat5"],
		"cat_features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "cat2", "cat4", "cat5"],
		"sample_rate": 1.0,
        "params": {
			'task_type': "CPU",
			'verbose': 200,
			'random_seed': 42,
			'allow_writing_files': False,

			'loss_function': 'Logloss',
			'eval_metric': 'Logloss',
			'boost_from_average': True,

			'iterations': 4000,
			'early_stopping_rounds': 300,
			'learning_rate': 2e-2,

			'depth': 6,
			'max_ctr_complexity': 4,
			'l2_leaf_reg': 5e2,
			'random_strength': 5.0,

			'posterior_sampling': True,
        }
	},
    "wo_2li": {
		"desc": "All \wo two less important, full data",
		"features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "time_since_last_click", "cat2", "cat4"],
		"cat_features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "cat2", "cat4"],
		"sample_rate": 1.0,
        "params": {
			'task_type': "CPU",
			'verbose': 200,
			'random_seed': 42,
			'allow_writing_files': False,

			'loss_function': 'Logloss',
			'eval_metric': 'Logloss',
			'boost_from_average': True,

			'iterations': 4000,
			'early_stopping_rounds': 300,
			'learning_rate': 2e-2,

			'depth': 6,
			'max_ctr_complexity': 4,
			'l2_leaf_reg': 5e2,
			'random_strength': 5.0,

			'posterior_sampling': True,
        }
	},
    "wo_3li": {
		"desc": "All \wo three less important, full data",
		"features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "time_since_last_click", "cat2"],
		"cat_features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "cat2"],
		"sample_rate": 1.0,
        "params": {
			'task_type': "CPU",
			'verbose': 200,
			'random_seed': 42,
			'allow_writing_files': False,

			'loss_function': 'Logloss',
			'eval_metric': 'Logloss',
			'boost_from_average': True,

			'iterations': 4000,
			'early_stopping_rounds': 250,
			'learning_rate': 2e-2,

			'depth': 6,
			'max_ctr_complexity': 4,
			'l2_leaf_reg': 5e2,
			'random_strength': 5.0,

			'posterior_sampling': True,
        }
	},
    "wo_4li": {
		"desc": "All \wo 4 less important, full data",
		"features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "time_since_last_click"],
		"cat_features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7"],
		"sample_rate": 1.0,
        "params": {
			'task_type': "CPU",
			'verbose': 200,
			'random_seed': 42,
			'allow_writing_files': False,

			'loss_function': 'Logloss',
			'eval_metric': 'Logloss',
			'boost_from_average': True,

			'iterations': 4000,
			'early_stopping_rounds': 250,
			'learning_rate': 2e-2,

			'depth': 6,
			'max_ctr_complexity': 4,
			'l2_leaf_reg': 5e2,
			'random_strength': 5.0,

			'posterior_sampling': True,
        }
	},
    "wo_5li": {
		"desc": "All \wo 5 less important, full data",
		"features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7"],
		"cat_features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7"],
		"sample_rate": 1.0,
        "params": {
			'task_type': "CPU",
			'verbose': 200,
			'random_seed': 42,
			'allow_writing_files': False,

			'loss_function': 'Logloss',
			'eval_metric': 'Logloss',
			'boost_from_average': True,

			'iterations': 4000,
			'early_stopping_rounds': 250,
			'learning_rate': 2e-2,

			'depth': 6,
			'max_ctr_complexity': 4,
			'l2_leaf_reg': 5e2,
			'random_strength': 5.0,

			'posterior_sampling': True,
        }
	},
    "wo_6li": {
		"desc": "All \wo 6 less important, full data",
		"features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign"],
		"cat_features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign"],
		"sample_rate": 1.0,
        "params": {
			'task_type': "CPU",
			'verbose': 200,
			'random_seed': 42,
			'allow_writing_files': False,

			'loss_function': 'Logloss',
			'eval_metric': 'Logloss',
			'boost_from_average': True,

			'iterations': 3000,
			'early_stopping_rounds': 200,
			'learning_rate': 2e-2,

			'depth': 4,
			'max_ctr_complexity': 4,
			'l2_leaf_reg': 5e2,
			'random_strength': 5.0,

			'posterior_sampling': True,
        }
	},
    "wo_7li": {
		"desc": "All \wo 7 less important, full data",
		"features": ["cat3", "cat8", "cat9", "cat6", "cat1"],
		"cat_features": ["cat3", "cat8", "cat9", "cat6", "cat1"],
		"sample_rate": 1.0,
        "params": {
			'task_type': "CPU",
			'verbose': 200,
			'random_seed': 42,
			'allow_writing_files': False,

			'loss_function': 'Logloss',
			'eval_metric': 'Logloss',
			'boost_from_average': True,

			'iterations': 3000,
			'early_stopping_rounds': 200,
			'learning_rate': 2e-2,

			'depth': 4,
			'max_ctr_complexity': 4,
			'l2_leaf_reg': 1e1,
			'random_strength': 1.0,

			'posterior_sampling': True,
        }
	},
    "wo_8li": {
		"desc": "All \wo 8 less important, full data",
		"features": ["cat3", "cat8", "cat9", "cat6"],
		"cat_features": ["cat3", "cat8", "cat9", "cat6"],
		"sample_rate": 1.0,
        "params": {
			'task_type': "CPU",
			'verbose': 200,
			'random_seed': 42,
			'allow_writing_files': False,

			'loss_function': 'Logloss',
			'eval_metric': 'Logloss',
			'boost_from_average': True,

			'iterations': 3000,
			'early_stopping_rounds': 200,
			'learning_rate': 2e-2,

			'depth': 4,
			'max_ctr_complexity': 4,
			'l2_leaf_reg': 1e1,
			'random_strength': 1.0,

			'posterior_sampling': True,
        }
	},
    "wo_9li": {
		"desc": "All \wo 9 less important, full data",
		"features": ["cat3", "cat8", "cat9"],
		"cat_features": ["cat3", "cat8", "cat9"],
		"sample_rate": 1.0,
        "params": {
			'task_type': "CPU",
			'verbose': 200,
			'random_seed': 42,
			'allow_writing_files': False,

			'loss_function': 'Logloss',
			'eval_metric': 'Logloss',
			'boost_from_average': True,

			'iterations': 3000,
			'early_stopping_rounds': 200,
			'learning_rate': 2e-2,

			'depth': 4,
			'max_ctr_complexity': 4,
			'l2_leaf_reg': 1e1,
			'random_strength': 1.0,

			'posterior_sampling': True,
        }
	},
    "wo_10li": {
		"desc": "All \wo 10 less important, full data",
		"features": ["cat3", "cat8"],
		"cat_features": ["cat3", "cat8"],
		"sample_rate": 1.0,
        "params": {
			'task_type': "CPU",
			'verbose': 200,
			'random_seed': 42,
			'allow_writing_files': False,

			'loss_function': 'Logloss',
			'eval_metric': 'Logloss',
			'boost_from_average': True,

			'iterations': 3000,
			'early_stopping_rounds': 200,
			'learning_rate': 2e-2,

			'depth': 4,
			'max_ctr_complexity': 4,
			'l2_leaf_reg': 1e1,
			'random_strength': 1.0,

			'posterior_sampling': True,
        }
	},
}

In [11]:
# del models
gc.collect()
models = train_model_pair(df_train, df_val, configs["all_features"])
df_bidding = apply_models_and_save(models, df_bidding, "all_features", configs["all_features"], filename="bidding_predictions.csv")
models['metrics']

Desc: All features, full data
Features: 12 | Cat features: 11
Train rows: 11527618
--> Fitting CVR Model (Clicks only)...
0:	learn: 0.3956298	test: 0.3910844	best: 0.3910844 (0)	total: 1.37s	remaining: 1h 31m 31s
200:	learn: 0.2887798	test: 0.2878887	best: 0.2878887 (200)	total: 2m 14s	remaining: 42m 17s
400:	learn: 0.2855020	test: 0.2855360	best: 0.2855360 (400)	total: 5m 5s	remaining: 45m 40s
600:	learn: 0.2842303	test: 0.2846659	best: 0.2846659 (600)	total: 7m 53s	remaining: 44m 38s
800:	learn: 0.2832811	test: 0.2840508	best: 0.2840508 (800)	total: 10m 48s	remaining: 43m 10s
1000:	learn: 0.2822968	test: 0.2834590	best: 0.2834590 (1000)	total: 13m 39s	remaining: 40m 54s
1200:	learn: 0.2817596	test: 0.2831883	best: 0.2831883 (1200)	total: 16m 34s	remaining: 38m 38s
1400:	learn: 0.2813835	test: 0.2830399	best: 0.2830399 (1400)	total: 19m 34s	remaining: 36m 19s
1600:	learn: 0.2811234	test: 0.2829813	best: 0.2829811 (1598)	total: 22m 33s	remaining: 33m 47s
1800:	learn: 0.2809360	test: 0.

{'cvr_best_score': 0.2829218548137628,
 'cvr_auc': 0.8513500260228819,
 'cvr_pr': 0.551811859518541,
 'ctr_best_score': 0.5993581652572881,
 'ctr_roc_auc': 0.6959822609792806,
 'ctr_pr_auc': 0.5733772355369997}

In [12]:
del models
gc.collect()
models = train_model_pair(df_train, df_val, configs["wo_10li"])
df_bidding = apply_models_and_save(models, df_bidding, "wo_10li", configs["wo_10li"], filename="bidding_predictions.csv")
models['metrics']

Desc: All \wo 10 less important, full data
Features: 2 | Cat features: 2
Train rows: 11527618
--> Fitting CVR Model (Clicks only)...
0:	learn: 0.3964912	test: 0.3917855	best: 0.3917855 (0)	total: 272ms	remaining: 13m 36s
200:	learn: 0.3241912	test: 0.3210645	best: 0.3210645 (200)	total: 36.6s	remaining: 8m 29s
400:	learn: 0.3234231	test: 0.3203201	best: 0.3203201 (400)	total: 1m 13s	remaining: 7m 54s
600:	learn: 0.3232259	test: 0.3201598	best: 0.3201593 (594)	total: 1m 50s	remaining: 7m 19s
800:	learn: 0.3231584	test: 0.3201375	best: 0.3201367 (798)	total: 2m 26s	remaining: 6m 43s
1000:	learn: 0.3231388	test: 0.3201414	best: 0.3201264 (894)	total: 3m 3s	remaining: 6m 7s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.3201264492
bestIteration = 894

Shrink model to first 895 iterations.
--> Fitting CTR Model...
0:	learn: 0.6516080	test: 0.6574708	best: 0.6574708 (0)	total: 705ms	remaining: 35m 14s
200:	learn: 0.6239093	test: 0.6270901	best: 0.6270901 (200)	total: 1m

{'cvr_best_score': 0.3201264491901804,
 'cvr_auc': 0.7944814016742335,
 'cvr_pr': 0.4280090923635219,
 'ctr_best_score': 0.6263334243926907,
 'ctr_roc_auc': 0.6424670542815696,
 'ctr_pr_auc': 0.5055894773992254}

In [13]:
del models
gc.collect()
models = train_model_pair(df_train, df_val, configs["wo_9li"])
df_bidding = apply_models_and_save(models, df_bidding, "wo_9li", configs["wo_9li"], filename="bidding_predictions.csv")
models['metrics']

Desc: All \wo 9 less important, full data
Features: 3 | Cat features: 3
Train rows: 11527618
--> Fitting CVR Model (Clicks only)...
0:	learn: 0.3966363	test: 0.3919551	best: 0.3919551 (0)	total: 269ms	remaining: 13m 26s
200:	learn: 0.3201566	test: 0.3173905	best: 0.3173905 (200)	total: 39.9s	remaining: 9m 15s
400:	learn: 0.3187970	test: 0.3162262	best: 0.3162262 (400)	total: 1m 19s	remaining: 8m 38s
600:	learn: 0.3183004	test: 0.3157744	best: 0.3157744 (600)	total: 2m	remaining: 7m 59s
800:	learn: 0.3180553	test: 0.3155544	best: 0.3155544 (800)	total: 2m 40s	remaining: 7m 20s
1000:	learn: 0.3179382	test: 0.3154682	best: 0.3154682 (1000)	total: 3m 20s	remaining: 6m 41s
1200:	learn: 0.3178795	test: 0.3154624	best: 0.3154619 (1199)	total: 4m 1s	remaining: 6m 1s
1400:	learn: 0.3178552	test: 0.3154731	best: 0.3154572 (1261)	total: 4m 41s	remaining: 5m 21s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.3154571531
bestIteration = 1261

Shrink model to first 1262 iteratio

{'cvr_best_score': 0.31545715306190525,
 'cvr_auc': 0.8037924414514195,
 'cvr_pr': 0.4496135544062697,
 'ctr_best_score': 0.6109996926824166,
 'ctr_roc_auc': 0.6737344773173115,
 'ctr_pr_auc': 0.5521456947105254}

In [14]:
del models
gc.collect()
models = train_model_pair(df_train, df_val, configs["wo_8li"])
df_bidding = apply_models_and_save(models, df_bidding, "wo_8li", configs["wo_8li"], filename="bidding_predictions.csv")
models['metrics']

Desc: All \wo 8 less important, full data
Features: 4 | Cat features: 4
Train rows: 11527618
--> Fitting CVR Model (Clicks only)...
0:	learn: 0.3966372	test: 0.3919587	best: 0.3919587 (0)	total: 278ms	remaining: 13m 54s
200:	learn: 0.3194962	test: 0.3166740	best: 0.3166740 (200)	total: 43.7s	remaining: 10m 8s
400:	learn: 0.3177402	test: 0.3150900	best: 0.3150900 (400)	total: 1m 28s	remaining: 9m 31s
600:	learn: 0.3170643	test: 0.3145082	best: 0.3145082 (600)	total: 2m 12s	remaining: 8m 49s
800:	learn: 0.3166765	test: 0.3141765	best: 0.3141765 (800)	total: 2m 56s	remaining: 8m 4s
1000:	learn: 0.3164771	test: 0.3140247	best: 0.3140240 (998)	total: 3m 41s	remaining: 7m 21s
1200:	learn: 0.3163927	test: 0.3139853	best: 0.3139853 (1192)	total: 4m 25s	remaining: 6m 37s
1400:	learn: 0.3163492	test: 0.3139955	best: 0.3139823 (1203)	total: 5m 9s	remaining: 5m 53s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.3139823358
bestIteration = 1203

Shrink model to first 1204 itera

{'cvr_best_score': 0.31398233580535295,
 'cvr_auc': 0.8063017630769922,
 'cvr_pr': 0.45449048792952346,
 'ctr_best_score': 0.6038926131609624,
 'ctr_roc_auc': 0.6876418391338526,
 'ctr_pr_auc': 0.5649107903465632}

In [15]:
del models
gc.collect()
models = train_model_pair(df_train, df_val, configs["wo_8li"])
df_bidding = apply_models_and_save(models, df_bidding, "wo_8li", configs["wo_8li"], filename="bidding_predictions.csv")
models['metrics']

Desc: All \wo 8 less important, full data
Features: 4 | Cat features: 4
Train rows: 11527618
--> Fitting CVR Model (Clicks only)...
0:	learn: 0.3966372	test: 0.3919587	best: 0.3919587 (0)	total: 333ms	remaining: 16m 37s
200:	learn: 0.3194962	test: 0.3166740	best: 0.3166740 (200)	total: 43.3s	remaining: 10m 2s
400:	learn: 0.3177402	test: 0.3150900	best: 0.3150900 (400)	total: 1m 27s	remaining: 9m 29s
600:	learn: 0.3170643	test: 0.3145082	best: 0.3145082 (600)	total: 2m 12s	remaining: 8m 47s
800:	learn: 0.3166765	test: 0.3141765	best: 0.3141765 (800)	total: 2m 55s	remaining: 8m 2s
1000:	learn: 0.3164771	test: 0.3140247	best: 0.3140240 (998)	total: 3m 40s	remaining: 7m 19s
1200:	learn: 0.3163927	test: 0.3139853	best: 0.3139853 (1192)	total: 4m 24s	remaining: 6m 36s
1400:	learn: 0.3163492	test: 0.3139955	best: 0.3139823 (1203)	total: 5m 9s	remaining: 5m 52s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.3139823358
bestIteration = 1203

Shrink model to first 1204 itera

{'cvr_best_score': 0.31398233580535295,
 'cvr_auc': 0.8063017630769922,
 'cvr_pr': 0.45449048792952346,
 'ctr_best_score': 0.6038926131609624,
 'ctr_roc_auc': 0.6876418391338526,
 'ctr_pr_auc': 0.5649107903465632}

In [16]:
del models
gc.collect()
models = train_model_pair(df_train, df_val, configs["wo_7li"])
df_bidding = apply_models_and_save(models, df_bidding, "wo_7li", configs["wo_7li"], filename="bidding_predictions.csv")
models['metrics']

Desc: All \wo 7 less important, full data
Features: 5 | Cat features: 5
Train rows: 11527618
--> Fitting CVR Model (Clicks only)...
0:	learn: 0.3962401	test: 0.3915125	best: 0.3915125 (0)	total: 325ms	remaining: 16m 14s
200:	learn: 0.2945783	test: 0.2915417	best: 0.2915417 (200)	total: 48.8s	remaining: 11m 19s
400:	learn: 0.2922262	test: 0.2897767	best: 0.2897767 (400)	total: 1m 37s	remaining: 10m 34s
600:	learn: 0.2913365	test: 0.2890400	best: 0.2890400 (600)	total: 2m 26s	remaining: 9m 44s
800:	learn: 0.2908644	test: 0.2886739	best: 0.2886739 (800)	total: 3m 14s	remaining: 8m 54s
1000:	learn: 0.2905136	test: 0.2884582	best: 0.2884582 (1000)	total: 4m 2s	remaining: 8m 5s
1200:	learn: 0.2903056	test: 0.2883522	best: 0.2883521 (1199)	total: 4m 51s	remaining: 7m 16s
1400:	learn: 0.2901795	test: 0.2883236	best: 0.2883192 (1353)	total: 5m 39s	remaining: 6m 27s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.288319237
bestIteration = 1353

Shrink model to first 1354 ite

{'cvr_best_score': 0.2883192370363666,
 'cvr_auc': 0.8450906036900183,
 'cvr_pr': 0.5284516098686733,
 'ctr_best_score': 0.600949477389749,
 'ctr_roc_auc': 0.6930792686666731,
 'ctr_pr_auc': 0.570190433423941}

In [17]:
del models
gc.collect()
models = train_model_pair(df_train, df_val, configs["wo_6li"])
df_bidding = apply_models_and_save(models, df_bidding, "wo_6li", configs["wo_6li"], filename="bidding_predictions.csv")
models['metrics']

Desc: All \wo 6 less important, full data
Features: 6 | Cat features: 6
Train rows: 11527618
--> Fitting CVR Model (Clicks only)...
0:	learn: 0.3962924	test: 0.3915621	best: 0.3915621 (0)	total: 389ms	remaining: 19m 26s
200:	learn: 0.2945924	test: 0.2914913	best: 0.2914913 (200)	total: 53.4s	remaining: 12m 23s
400:	learn: 0.2924699	test: 0.2898654	best: 0.2898654 (400)	total: 1m 46s	remaining: 11m 27s
600:	learn: 0.2916381	test: 0.2892254	best: 0.2892254 (600)	total: 2m 38s	remaining: 10m 32s
800:	learn: 0.2911043	test: 0.2888172	best: 0.2888172 (800)	total: 3m 30s	remaining: 9m 38s
1000:	learn: 0.2906009	test: 0.2884721	best: 0.2884721 (1000)	total: 4m 21s	remaining: 8m 42s
1200:	learn: 0.2902929	test: 0.2882855	best: 0.2882855 (1200)	total: 5m 13s	remaining: 7m 49s
1400:	learn: 0.2901130	test: 0.2881920	best: 0.2881918 (1399)	total: 6m 4s	remaining: 6m 56s
1600:	learn: 0.2900156	test: 0.2881567	best: 0.2881554 (1594)	total: 6m 55s	remaining: 6m 3s
1800:	learn: 0.2899404	test: 0.28814

{'cvr_best_score': 0.288140483302836,
 'cvr_auc': 0.8457715520813356,
 'cvr_pr': 0.5293335927000302,
 'ctr_best_score': 0.6008843266607058,
 'ctr_roc_auc': 0.6932594406116245,
 'ctr_pr_auc': 0.5703145846895972}

In [18]:
del models
gc.collect()
models = train_model_pair(df_train, df_val, configs["wo_6li"])
df_bidding = apply_models_and_save(models, df_bidding, "wo_6li", configs["wo_6li"], filename="bidding_predictions.csv")
models['metrics']

Desc: All \wo 6 less important, full data
Features: 6 | Cat features: 6
Train rows: 11527618
--> Fitting CVR Model (Clicks only)...
0:	learn: 0.3962924	test: 0.3915621	best: 0.3915621 (0)	total: 358ms	remaining: 17m 54s
200:	learn: 0.2945924	test: 0.2914913	best: 0.2914913 (200)	total: 49.1s	remaining: 11m 23s
400:	learn: 0.2924699	test: 0.2898654	best: 0.2898654 (400)	total: 1m 37s	remaining: 10m 33s
600:	learn: 0.2916381	test: 0.2892254	best: 0.2892254 (600)	total: 2m 25s	remaining: 9m 42s
800:	learn: 0.2911043	test: 0.2888172	best: 0.2888172 (800)	total: 3m 14s	remaining: 8m 52s
1000:	learn: 0.2906009	test: 0.2884721	best: 0.2884721 (1000)	total: 4m 1s	remaining: 8m 2s
1200:	learn: 0.2902929	test: 0.2882855	best: 0.2882855 (1200)	total: 4m 49s	remaining: 7m 13s
1400:	learn: 0.2901130	test: 0.2881920	best: 0.2881918 (1399)	total: 5m 36s	remaining: 6m 24s
1600:	learn: 0.2900156	test: 0.2881567	best: 0.2881554 (1594)	total: 6m 23s	remaining: 5m 35s
1800:	learn: 0.2899404	test: 0.288149

{'cvr_best_score': 0.288140483302836,
 'cvr_auc': 0.8457715520813356,
 'cvr_pr': 0.5293335927000302,
 'ctr_best_score': 0.6008843266607058,
 'ctr_roc_auc': 0.6932594406116245,
 'ctr_pr_auc': 0.5703145846895972}

In [19]:
del models
gc.collect()
models = train_model_pair(df_train, df_val, configs["wo_5li"])
df_bidding = apply_models_and_save(models, df_bidding, "wo_5li", configs["wo_5li"], filename="bidding_predictions.csv")
models['metrics']

Desc: All \wo 5 less important, full data
Features: 7 | Cat features: 7
Train rows: 11527618
--> Fitting CVR Model (Clicks only)...
0:	learn: 0.3957737	test: 0.3910718	best: 0.3910718 (0)	total: 571ms	remaining: 38m 2s
200:	learn: 0.2914536	test: 0.2890508	best: 0.2890508 (200)	total: 1m 14s	remaining: 23m 37s
400:	learn: 0.2888314	test: 0.2873750	best: 0.2873746 (399)	total: 3m 8s	remaining: 28m 10s
600:	learn: 0.2876756	test: 0.2866328	best: 0.2866328 (600)	total: 4m 59s	remaining: 28m 16s
800:	learn: 0.2869341	test: 0.2861656	best: 0.2861656 (800)	total: 6m 49s	remaining: 27m 13s
1000:	learn: 0.2862147	test: 0.2857895	best: 0.2857895 (1000)	total: 8m 38s	remaining: 25m 52s
1200:	learn: 0.2858249	test: 0.2856532	best: 0.2856532 (1200)	total: 10m 34s	remaining: 24m 39s
1400:	learn: 0.2855651	test: 0.2855819	best: 0.2855803 (1395)	total: 12m 31s	remaining: 23m 14s
1600:	learn: 0.2854272	test: 0.2856029	best: 0.2855741 (1446)	total: 14m 29s	remaining: 21m 43s
Stopped by overfitting dete

{'cvr_best_score': 0.28557413340354865,
 'cvr_auc': 0.8491208434699409,
 'cvr_pr': 0.5374329909087948,
 'ctr_best_score': 0.5996509009078619,
 'ctr_roc_auc': 0.6951559652437194,
 'ctr_pr_auc': 0.5727936535951487}

In [20]:
del models
gc.collect()
models = train_model_pair(df_train, df_val, configs["wo_4li"])
df_bidding = apply_models_and_save(models, df_bidding, "wo_4li", configs["wo_4li"], filename="bidding_predictions.csv")
models['metrics']

Desc: All \wo 4 less important, full data
Features: 8 | Cat features: 7
Train rows: 11527618
--> Fitting CVR Model (Clicks only)...
0:	learn: 0.3956880	test: 0.3909849	best: 0.3909849 (0)	total: 631ms	remaining: 42m 3s
200:	learn: 0.2912798	test: 0.2888396	best: 0.2888396 (200)	total: 1m 21s	remaining: 25m 45s
400:	learn: 0.2886225	test: 0.2871295	best: 0.2871295 (400)	total: 3m 17s	remaining: 29m 32s
600:	learn: 0.2876433	test: 0.2865058	best: 0.2865058 (600)	total: 5m 10s	remaining: 29m 17s
800:	learn: 0.2868597	test: 0.2860564	best: 0.2860564 (800)	total: 7m 4s	remaining: 28m 14s
1000:	learn: 0.2861230	test: 0.2857075	best: 0.2857075 (1000)	total: 8m 52s	remaining: 26m 36s
1200:	learn: 0.2857517	test: 0.2855753	best: 0.2855743 (1199)	total: 10m 51s	remaining: 25m 17s
1400:	learn: 0.2855297	test: 0.2855377	best: 0.2855366 (1380)	total: 12m 52s	remaining: 23m 53s
1600:	learn: 0.2853624	test: 0.2855359	best: 0.2855257 (1493)	total: 15m 1s	remaining: 22m 31s
Stopped by overfitting detec

{'cvr_best_score': 0.285525718393015,
 'cvr_auc': 0.8491622089527763,
 'cvr_pr': 0.537671177491971,
 'ctr_best_score': 0.5996422330121448,
 'ctr_roc_auc': 0.6953960130661547,
 'ctr_pr_auc': 0.5730331115896304}

In [21]:
del models
gc.collect()
models = train_model_pair(df_train, df_val, configs["wo_3li"])
df_bidding = apply_models_and_save(models, df_bidding, "wo_3li", configs["wo_3li"], filename="bidding_predictions.csv")
models['metrics']

Desc: All \wo three less important, full data
Features: 9 | Cat features: 8
Train rows: 11527618
--> Fitting CVR Model (Clicks only)...
0:	learn: 0.3955959	test: 0.3909022	best: 0.3909022 (0)	total: 824ms	remaining: 54m 56s
200:	learn: 0.2917856	test: 0.2893320	best: 0.2893320 (200)	total: 1m 43s	remaining: 32m 28s
400:	learn: 0.2887313	test: 0.2872312	best: 0.2872312 (400)	total: 4m 1s	remaining: 36m 9s
600:	learn: 0.2877126	test: 0.2865563	best: 0.2865563 (600)	total: 6m 22s	remaining: 36m 4s
800:	learn: 0.2868983	test: 0.2860259	best: 0.2860259 (800)	total: 8m 43s	remaining: 34m 49s
1000:	learn: 0.2860556	test: 0.2855625	best: 0.2855625 (1000)	total: 11m 5s	remaining: 33m 13s
1200:	learn: 0.2855915	test: 0.2853486	best: 0.2853486 (1200)	total: 13m 28s	remaining: 31m 24s
1400:	learn: 0.2852950	test: 0.2852548	best: 0.2852548 (1399)	total: 15m 57s	remaining: 29m 35s
1600:	learn: 0.2851195	test: 0.2852347	best: 0.2852307 (1579)	total: 18m 25s	remaining: 27m 37s
1800:	learn: 0.2849833	t

{'cvr_best_score': 0.2852307450495308,
 'cvr_auc': 0.8495954936171837,
 'cvr_pr': 0.5391808507850109,
 'ctr_best_score': 0.5994798405361487,
 'ctr_roc_auc': 0.6957002546417784,
 'ctr_pr_auc': 0.5731948783949901}

In [22]:
del models
gc.collect()
models = train_model_pair(df_train, df_val, configs["wo_2li"])
df_bidding = apply_models_and_save(models, df_bidding, "wo_2li", configs["wo_2li"], filename="bidding_predictions.csv")
models['metrics']

Desc: All \wo two less important, full data
Features: 10 | Cat features: 9
Train rows: 11527618
--> Fitting CVR Model (Clicks only)...
0:	learn: 0.3955734	test: 0.3910193	best: 0.3910193 (0)	total: 889ms	remaining: 59m 14s
200:	learn: 0.2887951	test: 0.2878357	best: 0.2878357 (200)	total: 1m 59s	remaining: 37m 29s
400:	learn: 0.2854553	test: 0.2854701	best: 0.2854701 (400)	total: 4m 40s	remaining: 41m 57s
600:	learn: 0.2841661	test: 0.2846231	best: 0.2846231 (600)	total: 7m 19s	remaining: 41m 26s
800:	learn: 0.2832395	test: 0.2839876	best: 0.2839876 (800)	total: 9m 59s	remaining: 39m 55s
1000:	learn: 0.2823364	test: 0.2834945	best: 0.2834945 (1000)	total: 12m 40s	remaining: 37m 59s
1200:	learn: 0.2818540	test: 0.2832544	best: 0.2832544 (1200)	total: 15m 23s	remaining: 35m 51s
1400:	learn: 0.2814766	test: 0.2830976	best: 0.2830976 (1400)	total: 18m 9s	remaining: 33m 40s
1600:	learn: 0.2812150	test: 0.2830412	best: 0.2830403 (1593)	total: 20m 57s	remaining: 31m 24s
1800:	learn: 0.2810024

{'cvr_best_score': 0.2829867939472197,
 'cvr_auc': 0.8511002225919221,
 'cvr_pr': 0.5517103088054506,
 'ctr_best_score': 0.5993667775154374,
 'ctr_roc_auc': 0.6959331255704772,
 'ctr_pr_auc': 0.5733413026250586}

In [ ]:
del models
gc.collect()
models = train_model_pair(df_train, df_val, configs["wo_1li"])
df_bidding = apply_models_and_save(models, df_bidding, "wo_1li", configs["wo_1li"], filename="bidding_predictions.csv")
models['metrics']

Desc: All \wo one less important, full data
Features: 11 | Cat features: 10
Train rows: 11527618
--> Fitting CVR Model (Clicks only)...
0:	learn: 0.3956260	test: 0.3910702	best: 0.3910702 (0)	total: 1.07s	remaining: 1h 11m 8s
200:	learn: 0.2887697	test: 0.2878446	best: 0.2878446 (200)	total: 2m 11s	remaining: 41m 34s
400:	learn: 0.2854634	test: 0.2854677	best: 0.2854677 (400)	total: 4m 59s	remaining: 44m 45s
600:	learn: 0.2841860	test: 0.2846339	best: 0.2846339 (600)	total: 7m 48s	remaining: 44m 10s
800:	learn: 0.2832850	test: 0.2840553	best: 0.2840553 (800)	total: 10m 37s	remaining: 42m 27s
1000:	learn: 0.2822837	test: 0.2834738	best: 0.2834738 (1000)	total: 13m 30s	remaining: 40m 29s
1200:	learn: 0.2817401	test: 0.2832098	best: 0.2832098 (1200)	total: 16m 30s	remaining: 38m 27s
1400:	learn: 0.2813539	test: 0.2830488	best: 0.2830488 (1400)	total: 19m 30s	remaining: 36m 10s
1600:	learn: 0.2810773	test: 0.2829846	best: 0.2829838 (1596)	total: 22m 28s	remaining: 33m 40s
1800:	learn: 0.28

In [ ]:
models["ctr_model"].get_feature_importance(prettified=True)

NameError: name 'models' is not defined

In [ ]:
models["cvr_model"].get_feature_importance(prettified=True)